### TypedDict

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import List, Optional, Annotated, TypedDict
from dotenv import load_dotenv
import json

load_dotenv()

# استخدام نموذج Gemini من Google
llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview", 
    temperature=0.6
)

# تعريف هيكل البيانات المطلوب باستخدام TypedDict
class Movie(TypedDict):
    title: Annotated[str, "Name of the movie"]
    release_year: Annotated[int, "Year the movie was released"]
    genres: Annotated[List[str], "List of genres the movie belongs to"]
    rating: Annotated[float, "Average rating of the movie on a scale of 1 to 10"]
    director: Annotated[str, "Name of the movie director"]
    box_office: Annotated[Optional[float], "Worldwide box office in millions USD, if known"]

# إضافة Structured Output للنموذج
structured_llm = llm.with_structured_output(Movie)

# استخدام النموذج للحصول على بيانات منظمة
result = structured_llm.invoke("Give me details of a movie named Inception")

# طباعة النتيجة
print("=== Movie Details ===")
print(f"Title: {result['title']}")
print(f"Release Year: {result['release_year']}")
print(f"Director: {result['director']}")
print(f"Genres: {', '.join(result['genres'])}")
print(f"Rating: {result['rating']}/10")
print(f"Box Office: ${result['box_office']} million" if result['box_office'] else "Box Office: Unknown")

print("\n=== Full Result ===")
print(result)
print(f"Type: {type(result)}")

=== Movie Details ===
Title: Inception
Release Year: 2010
Director: Christopher Nolan
Genres: Action, Sci-Fi, Adventure
Rating: 8.8/10
Box Office: $836848410 million

=== Full Result ===
{'title': 'Inception', 'release_year': 2010, 'genres': ['Action', 'Sci-Fi', 'Adventure'], 'rating': 8.8, 'director': 'Christopher Nolan', 'box_office': 836848410}
Type: <class 'dict'>


Pydantic

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from typing import List, Optional
import dotenv

dotenv.load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview", 
    temperature=0.6
)

class Movie(BaseModel):
    title: str = Field(..., description="Name of the movie")
    release_year: int = Field(..., description="Year the movie was released")
    genres: List[str] = Field(..., description="List of genres the movie belongs to")
    rating: float = Field(..., description="Average rating of the movie on a scale of 1 to 10")
    box_office: Optional[float] = Field(None, description="Worldwide box office in millions USD, if known")
    director: Optional[str] = Field(None, description="Director of the movie")

# إضافة Structured Output للنموذج
structured_llm = llm.with_structured_output(Movie)

# الحصول على بيانات الفيلم من النموذج
print("=== من النموذج ===")
ai_result = structured_llm.invoke("Give me details about the movie Inception.")
print(ai_result)

# التحقق من النوع
print(f"\nنوع النتيجة: {type(ai_result)}")
print(f"هل هو كائن Movie؟ {isinstance(ai_result, Movie)}")

# الوصول إلى الحقول
print(f"\nعنوان الفيلم: {ai_result.title}")
print(f"سنة الإصدار: {ai_result.release_year}")
print(f"التقييم: {ai_result.rating}/10")
print(f"الأنواع: {', '.join(ai_result.genres)}")

# مثال مع فيلم آخر
print("\n=== فيلم آخر ===")
another_movie = structured_llm.invoke("Tell me about The Matrix")
print(f"العنوان: {another_movie.title}")
print(f"المخرج: {another_movie.director if another_movie.director else 'غير معروف'}")

=== من النموذج ===
title='Inception' release_year=2010 genres=['Action', 'Sci-Fi', 'Adventure'] rating=8.8 box_office=836.8 director='Christopher Nolan'

نوع النتيجة: <class '__main__.Movie'>
هل هو كائن Movie؟ True

عنوان الفيلم: Inception
سنة الإصدار: 2010
التقييم: 8.8/10
الأنواع: Action, Sci-Fi, Adventure

=== فيلم آخر ===
العنوان: The Matrix
المخرج: Lana Wachowski, Lilly Wachowski


JSON_Schema

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
import dotenv

dotenv.load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0)

# ✅ Correct JSON Schema for LangChain / Google Generative AI function calling
movie_json_schema = {
    "name": "movie_schema",
    "description": "Schema for extracting movie information",
    "parameters": {
        "type": "object",
        "properties": {
            "title": {
                "type": "string",
                "description": "Name of the movie"
            },
            "release_year": {
                "type": "integer",
                "description": "Year the movie was released"
            },
            "genres": {
                "type": "array",
                "items": {"type": "string"},
                "description": "List of genres the movie belongs to"
            },
            "rating": {
                "type": "number",
                "description": "Average rating of the movie on a scale of 1 to 10"
            },
            "box_office": {
                "type": ["number", "null"],
                "description": "Worldwide box office in millions USD, if known"
            }
        },
        "required": ["title", "release_year", "genres", "rating", "box_office"]
    }
}

structured_llm = llm.with_structured_output(movie_json_schema)

result = structured_llm.invoke("Give me details about the movie wolf of wall street.")
print(result)


{'title': 'The Wolf of Wall Street', 'director': 'Martin Scorsese', 'release_year': 2013, 'cast': ['Leonardo DiCaprio', 'Jonah Hill', 'Margot Robbie', 'Matthew McConaughey'], 'genres': ['Biography', 'Comedy', 'Crime'], 'rating': 8.2, 'plot': 'Based on the true story of Jordan Belfort, from his rise to a wealthy stock-broker living the high life to his fall involving crime, corruption and the federal government.'}
